[![Labellerr](https://storage.googleapis.com/labellerr-cdn/%200%20Labellerr%20template/notebook.webp)](https://www.labellerr.com)

# **YOLO and RTDETR comparison on small object detection using SAHI**

---

[![labellerr](https://img.shields.io/badge/Labellerr-BLOG-black.svg)](https://www.labellerr.com/blog/<BLOG_NAME>)
[![Youtube](https://img.shields.io/badge/Labellerr-YouTube-b31b1b.svg)](https://www.youtube.com/@Labellerr)
[![Github](https://img.shields.io/badge/Labellerr-GitHub-green.svg)](https://github.com/Labellerr/Hands-On-Learning-in-Computer-Vision)
[![Scientific Paper](https://img.shields.io/badge/Official-Paper-blue.svg)](<PAPER LINK>)

## YOLOv8 Baseline Inference Pipeline

### Model Initialization
A pretrained **YOLOv8 Medium (`yolov8m`) model** is loaded from the Ultralytics framework.  
This model is trained on the **COCO dataset**, which contains 80 object categories. The model will be used to perform object detection on the dataset images.

---

### Dataset and Output Directory Setup
The code defines two important paths:

- **Image directory**: location where all dataset images are stored.
- **Output directory**: location where the prediction results will be saved.

The output directory is created automatically if it does not already exist. This ensures the pipeline runs without errors when saving results.

---

### Image Collection
All `.jpg` images inside the dataset folder are collected and sorted.  
Sorting ensures that images are processed in a **consistent and reproducible order**, which is important when assigning sequential image IDs later for evaluation.

---

### Model Inference
The YOLO model performs detection on all images in the dataset with the following settings:

- **Confidence threshold: 0.15**  
  A relatively low threshold is used so that small or distant objects are not filtered out prematurely.

- **Class filter: COCO class 2 (car)**  
  Detection is restricted to only the **car category**, ignoring all other object classes.

- **Device: CPU**  
  Inference is executed on the CPU instead of GPU.

During inference, the model outputs:

- Bounding box coordinates
- Detection confidence scores
- Predicted class labels

---

### Saving Visual Predictions
For every processed image, the detected bounding boxes are drawn directly on the image.  
These annotated images are saved to the output directory so that detection quality can be visually inspected.

Each saved image contains:

- Bounding boxes around detected cars
- Confidence scores associated with each detection

---

### Preparing COCO-Format Predictions
To evaluate model performance using COCO metrics, the detections must be converted into **COCO prediction format**.

Each prediction record contains:

- **image_id** — identifier of the processed image
- **category_id** — class label (mapped to the dataset class index)
- **bbox** — bounding box in COCO format `[x, y, width, height]`
- **score** — confidence value of the detection

Because YOLO outputs bounding boxes in the format `[x1, y1, x2, y2]`, the width and height are calculated as:

- `width = x2 - x1`
- `height = y2 - y1`

Images that contain **no detections** are skipped to avoid adding empty entries.

---

### Exporting Prediction Results
All detection records are stored in a list and exported into a **JSON file**.

This file serves as the **prediction file for evaluation** and will later be used to compute metrics such as:

- Precision
- Recall
- Mean Average Precision (mAP)

The final JSON file contains the complete set of predicted bounding boxes across the dataset.

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import json


model = YOLO("yolov8m.pt")

image_dir = Path("dataset/images")
output_dir = Path("outputs/normal_yolo")
output_dir.mkdir(parents=True, exist_ok=True)

image_paths = sorted(image_dir.glob("*.jpg"))

# Running inference on coco class 2 as it was a by default car class
results = model(
    [str(p) for p in image_paths],
    conf=0.15,
    classes=[2],
    device="cpu"
)



for img_path, r in zip(image_paths, results):
    r.save(filename=str(output_dir / img_path.name))

print("Normal YOLOv8 (car-only) predictions saved")

#coco json

coco_preds = []
image_id = 1  

for r in results:
    if r.boxes is None:
        image_id += 1
        continue

    for box in r.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        coco_preds.append({
            "image_id": image_id,
            "category_id": 0,   
            "bbox": [x1, y1, x2 - x1, y2 - y1],
            "score": float(box.conf[0])
        })

    image_id += 1

with open("normal_yolo_preds.json", "w") as f:
    json.dump(coco_preds, f)

print("Normal YOLOv8 predictions JSON saved")



0: 640x640 13 cars, 428.6ms
1: 640x640 11 cars, 428.6ms
2: 640x640 3 cars, 428.6ms
Speed: 5.0ms preprocess, 428.6ms inference, 0.8ms postprocess per image at shape (1, 3, 640, 640)
Normal YOLOv8 (car-only) predictions saved
Normal YOLOv8 predictions JSON saved


## YOLOv8 + SAHI Sliced Inference Pipeline

### Detection Model Initialization
A **SAHI detection wrapper** is created around the pretrained **YOLOv8 Medium model** using `AutoDetectionModel`.  
This allows YOLO to perform **sliced inference**, where large images are divided into smaller tiles before detection.

Key configuration:

- **Model type**: YOLOv8  
- **Model weights**: `yolov8m.pt`  
- **Confidence threshold**: `0.15` to retain lower-confidence detections that often correspond to small objects  
- **Device**: CPU execution  

This wrapper enables SAHI to internally handle slicing, prediction, and merging of detections.

---

### Dataset and Output Structure
Two paths are defined:

- **Dataset directory** → contains the evaluation images.
- **Output directory** → stores images with SAHI detection overlays.

The output folder is created automatically if it does not exist to ensure prediction visuals can be exported without errors.

---

### Class Filtering
The model is restricted to detect **only the COCO "car" category (class ID 2)**.

A list of all other COCO class IDs is generated and excluded during inference.  
This ensures the pipeline only returns **car detections**, matching the dataset labeling scheme used in evaluation.

---

### Sliced Inference Strategy
Each image is processed using **SAHI sliced prediction**, which improves detection of small objects.

The image is divided into overlapping tiles with the following configuration:

- **Slice size**: `512 × 512`
- **Overlap ratio**: `0.25` in both height and width

Overlapping slices are important because objects located near tile boundaries could otherwise be partially visible and missed by the detector.

SAHI automatically:

1. Splits the image into slices  
2. Runs YOLO inference on each slice  
3. Merges predictions back into full-image coordinates  
4. Applies duplicate suppression for overlapping detections

---

### Visual Prediction Export
For every processed image, SAHI exports a **visualized prediction image** containing:

- Bounding boxes
- Detected object labels
- Confidence scores

These images are saved to the output directory so the detection quality can be visually inspected.

---

### Prediction Extraction
After inference, all detected objects are retrieved from the **object prediction list**.

Each detection provides:

- Bounding box coordinates
- Detection confidence score
- Predicted class information

Bounding boxes are automatically converted to **COCO-compatible `[x, y, width, height]` format** using the built-in SAHI utility.

---

### COCO Prediction Formatting
Each detection is stored as a dictionary containing:

- **image_id** → sequential identifier for the processed image  
- **category_id** → mapped dataset class ID (car)  
- **bbox** → bounding box in `[x, y, width, height]` format  
- **score** → prediction confidence  

The `image_id` counter is incremented after processing each image to maintain correct alignment with the dataset ground truth.

---

### Exporting Prediction Results
All collected predictions are stored in a list and exported to a **JSON file**.

This JSON file is structured according to the **COCO detection format**, making it compatible with evaluation tools that compute metrics such as:

- Precision
- Recall
- Mean Average Precision (mAP)

The exported file contains all detections generated using **YOLOv8 combined with SAHI sliced inference**.

In [ ]:
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from pathlib import Path
import json

detection_model = AutoDetectionModel.from_pretrained(
    model_type="yolov8",
    model_path="yolov8m.pt",
    confidence_threshold=0.15,
    device="cpu"
)

image_dir = Path("dataset/images")
output_dir = Path("outputs/sahi_yolo")
output_dir.mkdir(parents=True, exist_ok=True)


EXCLUDE_IDS = [i for i in range(80) if i != 2]

coco_preds = []
image_id = 1  # MUST match GT image_id order

for img_path in sorted(image_dir.glob("*.jpg")):
    result = get_sliced_prediction(
        image=str(img_path),
        detection_model=detection_model,
        slice_height=512,
        slice_width=512,
        overlap_height_ratio=0.25,
        overlap_width_ratio=0.25,
        exclude_classes_by_id=EXCLUDE_IDS
    )

    
    result.export_visuals(
        export_dir=str(output_dir),
        file_name=img_path.stem
    )

    
    for obj in result.object_prediction_list:
        coco_preds.append({
            "image_id": image_id,
            "category_id": 0,  
            "bbox": obj.bbox.to_xywh(),
            "score": obj.score.value
        })

    image_id += 1

with open("sahi_yolo_preds.json", "w") as f:
    json.dump(coco_preds, f)

print("YOLOv8 + SAHI (car-only) predictions saved + JSON exported")


Performing prediction on 66 slices.
Performing prediction on 80 slices.
Performing prediction on 80 slices.
YOLOv8 + SAHI (car-only) predictions saved + JSON exported


# Object Detection Evaluation Script


## Step-by-Step Breakdown

### 1. Load the Data
Three JSON files are loaded:
- **Ground Truth** — the correct bounding boxes for each image
- **YOLO Predictions** — boxes detected by standard YOLO
- **SAHI Predictions** — boxes detected by YOLO with slicing enabled

### 2. Convert Bounding Box Format
Boxes come in `[x, y, width, height]` format (COCO standard). They get converted to `[x1, y1, x2, y2]` so overlap can be calculated properly.

### 3. Calculate IoU (Intersection over Union)
IoU measures how much two boxes overlap:
- **0** = no overlap at all
- **1** = perfect overlap

A predicted box is considered a correct detection only if its IoU with a ground truth box is **≥ 0.5**.

### 4. Group Boxes by Image
All boxes are organized by image ID so the script can quickly find which boxes belong to which image.

### 5. Evaluate Predictions
For each image, the script counts:

| Metric | Meaning |
|--------|---------|
| **TP** (True Positive) | Prediction matched a ground truth box |
| **FP** (False Positive) | Prediction had no matching ground truth box |
| **FN** (False Negative) | Ground truth box was missed by the model |

From these, two scores are calculated:

- **Precision** = `TP / (TP + FP)` → Of all predictions made, how many were actually correct?
- **Recall** = `TP / (TP + FN)` → Of all actual cars, how many were successfully found?

### 6. Print Results
A side-by-side comparison is printed for every image, showing how plain YOLO and YOLO+SAHI differ in TP, FP, FN, Precision, and Recall.

---

## Goal
To determine whether SAHI's image-slicing technique improves detection — especially for **small or densely packed objects** where standard YOLO typically underperforms.

In [ ]:
import json
from collections import defaultdict
from pathlib import Path


GT_JSON_PATH = Path(r"give the file path")
YOLO_JSON_PATH = Path(r"give the file path")
SAHI_JSON_PATH = Path(r"give the file path")


IOU_THRESHOLD = 0.5



def xywh_to_xyxy(box):
    x, y, w, h = box
    return [x, y, x + w, y + h]

def iou(box1, box2):
    x1, y1, x2, y2 = box1
    x1g, y1g, x2g, y2g = box2

    xi1, yi1 = max(x1, x1g), max(y1, y1g)
    xi2, yi2 = min(x2, x2g), min(y2, y2g)

    inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    box1_area = (x2 - x1) * (y2 - y1)
    box2_area = (x2g - x1g) * (y2g - y1g)

    union = box1_area + box2_area - inter_area
    return inter_area / union if union > 0 else 0



with open(GT_JSON_PATH, "r") as f:
    gt = json.load(f)

with open(YOLO_JSON_PATH, "r") as f:
    yolo_preds = json.load(f)

with open(SAHI_JSON_PATH, "r") as f:
    sahi_preds = json.load(f)



gt_by_image = defaultdict(list)
for ann in gt["annotations"]:
    gt_by_image[ann["image_id"]].append(xywh_to_xyxy(ann["bbox"]))

yolo_by_image = defaultdict(list)
for p in yolo_preds:
    yolo_by_image[p["image_id"]].append(xywh_to_xyxy(p["bbox"]))

sahi_by_image = defaultdict(list)
for p in sahi_preds:
    sahi_by_image[p["image_id"]].append(xywh_to_xyxy(p["bbox"]))



def evaluate(gt_boxes, pred_boxes):
    matched_gt = set()
    tp = 0

    for pb in pred_boxes:
        for i, gb in enumerate(gt_boxes):
            if i in matched_gt:
                continue
            if iou(pb, gb) >= IOU_THRESHOLD:
                tp += 1
                matched_gt.add(i)
                break

    fp = len(pred_boxes) - tp
    fn = len(gt_boxes) - tp

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    return tp, fp, fn, precision, recall



print("\n================ PER IMAGE ANALYSIS ================\n")

for img in gt["images"]:
    img_id = img["id"]
    name = img["file_name"]

    gt_boxes = gt_by_image.get(img_id, [])
    yolo_boxes = yolo_by_image.get(img_id, [])
    sahi_boxes = sahi_by_image.get(img_id, [])

    y_tp, y_fp, y_fn, y_p, y_r = evaluate(gt_boxes, yolo_boxes)
    s_tp, s_fp, s_fn, s_p, s_r = evaluate(gt_boxes, sahi_boxes)

    print(f"Image: {name}")
    print(f"  GT cars: {len(gt_boxes)}")

    print("  Normal YOLO:")
    print(f"    TP={y_tp}, FP={y_fp}, FN={y_fn}")
    print(f"    Precision={y_p:.3f}, Recall={y_r:.3f}")

    print("  YOLO + SAHI:")
    print(f"    TP={s_tp}, FP={s_fp}, FN={s_fn}")
    print(f"    Precision={s_p:.3f}, Recall={s_r:.3f}")

    print("-" * 55)

print("\nComparison complete")



================ PER IMAGE ANALYSIS ================

Image: car3.jpg
  GT cars: 55
  Normal YOLO:
    TP=0, FP=13, FN=55
    Precision=0.000, Recall=0.000
  YOLO + SAHI:
    TP=0, FP=38, FN=55
    Precision=0.000, Recall=0.000
-------------------------------------------------------
Image: car2.jpg
  GT cars: 22
  Normal YOLO:
    TP=7, FP=4, FN=15
    Precision=0.636, Recall=0.318
  YOLO + SAHI:
    TP=19, FP=15, FN=3
    Precision=0.559, Recall=0.864
-------------------------------------------------------
Image: car1.jpg
  GT cars: 51
  Normal YOLO:
    TP=0, FP=3, FN=51
    Precision=0.000, Recall=0.000
  YOLO + SAHI:
    TP=0, FP=3, FN=51
    Precision=0.000, Recall=0.000
-------------------------------------------------------

✔ Comparison complete


# RT-DETR Car Detection Script

## Overview

This script runs **RT-DETR** (a real-time detection transformer model) on a folder of images to detect cars, saves the annotated output images, and exports the predictions in COCO JSON format.

---

## Step-by-Step Breakdown

### 1. Load the Model
```python
model = YOLO("rtdetr-l.pt")
```
Loads the RT-DETR large model using Ultralytics' YOLO interface. Despite the `YOLO()` call, RT-DETR is a transformer-based detector — Ultralytics just uses the same API for both.

### 2. Set Up Paths
- **Input** — reads all `.jpg` images from `dataset/images/`
- **Output** — saves annotated images to `outputs/normal_rtdetr/` (creates the folder if it doesn't exist)

### 3. Run Inference
```python
results = model([...], conf=0.15, classes=[2], device="cpu")
```

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `conf` | `0.15` | Only keep detections with ≥ 15% confidence |
| `classes` | `[2]` | Only detect class 2 — **cars** in COCO |
| `device` | `cpu` | Run on CPU instead of GPU |

All images are passed in one batch for efficiency.

### 4. Save Annotated Images
Each result is saved as an image with bounding boxes drawn on it, into the output folder.

### 5. Export Predictions to COCO JSON
The predictions are converted into COCO format and saved as `normal_rtdetr_preds.json`. Each entry looks like:

```json
{
  "image_id": 1,
  "category_id": 0,
  "bbox": [x, y, width, height],
  "score": 0.87
}
```

Note that `image_id` is assigned by simple incrementing (1, 2, 3...) based on the order images are processed.

---

## Output
- Annotated `.jpg` images in `outputs/normal_rtdetr/`
- `normal_rtdetr_preds.json` — predictions ready to be used in evaluation scripts (like the comparison script)

In [ ]:
from ultralytics import YOLO
from pathlib import Path
import json


model = YOLO("rtdetr-l.pt")

image_dir = Path("dataset/images")
output_dir = Path("outputs/normal_rtdetr")
output_dir.mkdir(parents=True, exist_ok=True)

image_paths = sorted(image_dir.glob("*.jpg"))


results = model(
    [str(p) for p in image_paths],
    conf=0.15,
    classes=[2],
    device="cpu"
)


for img_path, r in zip(image_paths, results):
    r.save(filename=str(output_dir / img_path.name))


coco_preds = []
image_id = 1

for r in results:
    if r.boxes is None:
        image_id += 1
        continue

    for box in r.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        coco_preds.append({
            "image_id": image_id,
            "category_id": 0,  
            "bbox": [x1, y1, x2 - x1, y2 - y1],
            "score": float(box.conf[0])
        })

    image_id += 1

with open("normal_rtdetr_preds.json", "w") as f:
    json.dump(coco_preds, f)

print("Normal RT-DETR (car-only) saved + JSON exported")



0: 640x640 38 cars, 2880.9ms
1: 640x640 46 cars, 2880.9ms
2: 640x640 (no detections), 2880.9ms
Speed: 5.9ms preprocess, 2880.9ms inference, 1.7ms postprocess per image at shape (1, 3, 640, 640)
Normal RT-DETR (car-only) saved + JSON exported


# RT-DETR + SAHI Car Detection Script

## Overview

This script runs **RT-DETR** with **SAHI** (Slicing Aided Hyper Inference) to detect cars in images. SAHI improves detection of small or distant objects by slicing each image into overlapping tiles and running inference on each tile separately before merging the results.

---

## Step-by-Step Breakdown

### 1. Load the Model
```python
detection_model = AutoDetectionModel.from_pretrained(
    model_type="rtdetr",
    model_path="rtdetr-l.pt",
    confidence_threshold=0.15,
    device="cpu"
)
```
Loads the RT-DETR model through SAHI's wrapper, which allows it to be used with the slicing pipeline. Confidence threshold is set to 15%.

### 2. Set Up Paths
- **Input** — reads all `.jpg` images from `dataset/images/`
- **Output** — saves annotated images to `outputs/sahi_rtdetr/`

### 3. Filter to Cars Only
```python
EXCLUDE_IDS = [i for i in range(80) if i != 2]
```
Builds a list of all 80 COCO class IDs except class **2 (car)**, then passes it to SAHI to exclude everything that isn't a car.

### 4. Run Sliced Inference (Per Image)
```python
result = get_sliced_prediction(
    image=str(img_path),
    detection_model=detection_model,
    slice_height=512,
    slice_width=512,
    overlap_height_ratio=0.25,
    overlap_width_ratio=0.25,
    ...
)
```
Each image is cut into **512×512 tiles** with a **25% overlap** on both axes. RT-DETR runs on every tile, and SAHI automatically merges the results back together using NMS (Non-Maximum Suppression) to remove duplicate boxes.

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `slice_height / width` | `512` | Size of each tile in pixels |
| `overlap_height / width_ratio` | `0.25` | 25% overlap between tiles to avoid missing objects at edges |

### 5. Save Annotated Images
The result with bounding boxes drawn is exported to the output folder for each image.

### 6. Export Predictions to COCO JSON
Each detected object is saved in COCO format to `sahi_rtdetr_preds.json`:

```json
{
  "image_id": 1,
  "category_id": 0,
  "bbox": [x, y, width, height],
  "score": 0.82
}
```

---

## Key Difference from Plain RT-DETR

| | Plain RT-DETR | RT-DETR + SAHI |
|--|---------------|----------------|
| Input to model | Full image | 512×512 tiles |
| Small object detection | Weaker | Stronger |
| Speed | Faster | Slower |
| Output format | Same COCO JSON | Same COCO JSON |

SAHI is especially useful when cars appear small in the frame — such as aerial or wide-angle shots — where a full-image pass would miss them.

In [ ]:
from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction
from pathlib import Path
import json

detection_model = AutoDetectionModel.from_pretrained(
    model_type="rtdetr",
    model_path="rtdetr-l.pt",
    confidence_threshold=0.15,
    device="cpu"
)

image_dir = Path("dataset/images")
output_dir = Path("outputs/sahi_rtdetr")
output_dir.mkdir(parents=True, exist_ok=True)


EXCLUDE_IDS = [i for i in range(80) if i != 2]

coco_preds = []
image_id = 1

for img_path in sorted(image_dir.glob("*.jpg")):
    result = get_sliced_prediction(
        image=str(img_path),
        detection_model=detection_model,
        slice_height=512,
        slice_width=512,
        overlap_height_ratio=0.25,
        overlap_width_ratio=0.25,
        exclude_classes_by_id=EXCLUDE_IDS
    )

    
    result.export_visuals(
        export_dir=str(output_dir),
        file_name=img_path.stem
    )

   
    for obj in result.object_prediction_list:
        coco_preds.append({
            "image_id": image_id,
            "category_id": 0, 
            "bbox": obj.bbox.to_xywh(),
            "score": obj.score.value
        })

    image_id += 1

with open("sahi_rtdetr_preds.json", "w") as f:
    json.dump(coco_preds, f)

print("RT-DETR + SAHI (car-only) saved + JSON exported")


Performing prediction on 66 slices.
Performing prediction on 80 slices.
Performing prediction on 80 slices.
RT-DETR + SAHI (car-only) saved + JSON exported


# Single Image Model Comparison Script

## Overview

This script compares **plain RT-DETR** vs **RT-DETR + SAHI** on a **single specific image**, evaluating which model detects cars more accurately using Precision and Recall.

---

## Step-by-Step Breakdown

### 1. Configuration
```python
IMAGE_ID = 2
IOU_THRESHOLD = 0.5
```
Unlike the full evaluation script that loops over all images, this one targets **one image by ID**. A 0.5 IoU threshold is used to decide if a prediction counts as correct.

### 2. Load the Data
Three JSON files are loaded:
- **Ground Truth** — the correct bounding boxes
- **PRED1** — predictions from plain RT-DETR
- **PRED2** — predictions from RT-DETR + SAHI

### 3. Filter Boxes for the Target Image
Instead of grouping all images, the script filters each JSON directly for entries where `image_id == 2`, giving three flat lists of boxes for that one image.

### 4. Box Conversion & IoU
Same helpers as the full evaluation script:
- `xywh_to_xyxy` converts COCO format boxes to corner format
- `iou()` measures overlap between two boxes (0 = no overlap, 1 = perfect)

### 5. Evaluate Both Models
The `evaluate()` function counts for each model:

| Metric | Meaning |
|--------|---------|
| **TP** | Predicted box matched a ground truth box (IoU ≥ 0.5) |
| **FP** | Predicted box with no matching ground truth |
| **FN** | Ground truth box that was missed |

Then calculates:
- **Precision** = `TP / (TP + FP)` → How many predictions were correct?
- **Recall** = `TP / (TP + FN)` → How many actual cars were found?

### 6. Print Results
Prints a focused comparison for the single image showing TP/FP/FN and Recall for both models.

---

## Difference from the Full Evaluation Script

| | Full Script | This Script |
|--|-------------|-------------|
| Scope | All images | One image (ID = 2) |
| Use case | Overall benchmark | Debugging a specific image |
| Output | Per-image table | Single focused comparison |

This is useful for **inspecting a hard case** — for example, an image where one model clearly outperforms the other.

In [ ]:
import json
from collections import defaultdict
from pathlib import Path

GT_JSON_PATH = Path(r"file path")
PRED1_JSON_PATH = Path(r"file path")
PRED2_JSON_PATH = Path(r"file path")


IMAGE_ID = 2
IOU_THRESHOLD = 0.5



def xywh_to_xyxy(box):
    x, y, w, h = box
    return [x, y, x + w, y + h]

def iou(box1, box2):
    x1, y1, x2, y2 = box1
    x1g, y1g, x2g, y2g = box2

    xi1, yi1 = max(x1, x1g), max(y1, y1g)
    xi2, yi2 = min(x2, x2g), min(y2, y2g)

    inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    box1_area = (x2 - x1) * (y2 - y1)
    box2_area = (x2g - x1g) * (y2g - y1g)

    union = box1_area + box2_area - inter_area
    return inter_area / union if union > 0 else 0



with open(GT_JSON_PATH, "r") as f:
    gt = json.load(f)

with open(PRED1_JSON_PATH, "r") as f:
    preds1 = json.load(f)

with open(PRED2_JSON_PATH, "r") as f:
    preds2 = json.load(f)



gt_boxes = [
    xywh_to_xyxy(a["bbox"])
    for a in gt["annotations"]
    if a["image_id"] == IMAGE_ID
]

pred1_boxes = [
    xywh_to_xyxy(p["bbox"])
    for p in preds1
    if p["image_id"] == IMAGE_ID
]

pred2_boxes = [
    xywh_to_xyxy(p["bbox"])
    for p in preds2
    if p["image_id"] == IMAGE_ID
]



def evaluate(gt_boxes, pred_boxes):
    matched_gt = set()
    tp = 0

    for pb in pred_boxes:
        for i, gb in enumerate(gt_boxes):
            if i in matched_gt:
                continue
            if iou(pb, gb) >= IOU_THRESHOLD:
                tp += 1
                matched_gt.add(i)
                break

    fp = len(pred_boxes) - tp
    fn = len(gt_boxes) - tp

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    return tp, fp, fn, precision, recall



p1_tp, p1_fp, p1_fn, p1_p, p1_r = evaluate(gt_boxes, pred1_boxes)
p2_tp, p2_fp, p2_fn, p2_p, p2_r = evaluate(gt_boxes, pred2_boxes)

print("\n=========== IMAGE  ANALYSIS ===========\n")
print(f"GT cars: {len(gt_boxes)}\n")

print("Model 1 (Normal):")
print(f"  TP={p1_tp}, FP={p1_fp}, FN={p1_fn}")
print(f"   Recall={p1_r:.3f}\n")

print("Model 2 (With SAHI):")
print(f"  TP={p2_tp}, FP={p2_fp}, FN={p2_fn}")
print(f"   Recall={p2_r:.3f}")

print("\n✔ Image 2 comparison complete")



=========== IMAGE  ANALYSIS ===========

GT cars: 22

Model 1 (Normal):
  TP=7, FP=4, FN=15
   Recall=0.318

Model 2 (With SAHI):
  TP=19, FP=15, FN=3
   Recall=0.864

✔ Image 2 comparison complete


In [ ]:
import json
from collections import defaultdict
from pathlib import Path

GT_JSON_PATH = Path(r"file path")
PRED1_JSON_PATH = Path(r"file path")
PRED2_JSON_PATH = Path(r"file path")


IMAGE_ID = 2
IOU_THRESHOLD = 0.5



def xywh_to_xyxy(box):
    x, y, w, h = box
    return [x, y, x + w, y + h]

def iou(box1, box2):
    x1, y1, x2, y2 = box1
    x1g, y1g, x2g, y2g = box2

    xi1, yi1 = max(x1, x1g), max(y1, y1g)
    xi2, yi2 = min(x2, x2g), min(y2, y2g)

    inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    box1_area = (x2 - x1) * (y2 - y1)
    box2_area = (x2g - x1g) * (y2g - y1g)

    union = box1_area + box2_area - inter_area
    return inter_area / union if union > 0 else 0



with open(GT_JSON_PATH, "r") as f:
    gt = json.load(f)

with open(PRED1_JSON_PATH, "r") as f:
    preds1 = json.load(f)

with open(PRED2_JSON_PATH, "r") as f:
    preds2 = json.load(f)



gt_boxes = [
    xywh_to_xyxy(a["bbox"])
    for a in gt["annotations"]
    if a["image_id"] == IMAGE_ID
]

pred1_boxes = [
    xywh_to_xyxy(p["bbox"])
    for p in preds1
    if p["image_id"] == IMAGE_ID
]

pred2_boxes = [
    xywh_to_xyxy(p["bbox"])
    for p in preds2
    if p["image_id"] == IMAGE_ID
]



def evaluate(gt_boxes, pred_boxes):
    matched_gt = set()
    tp = 0

    for pb in pred_boxes:
        for i, gb in enumerate(gt_boxes):
            if i in matched_gt:
                continue
            if iou(pb, gb) >= IOU_THRESHOLD:
                tp += 1
                matched_gt.add(i)
                break

    fp = len(pred_boxes) - tp
    fn = len(gt_boxes) - tp

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    return tp, fp, fn, precision, recall



p1_tp, p1_fp, p1_fn, p1_p, p1_r = evaluate(gt_boxes, pred1_boxes)
p2_tp, p2_fp, p2_fn, p2_p, p2_r = evaluate(gt_boxes, pred2_boxes)

print("\n=========== IMAGE 2 ANALYSIS ===========\n")
print(f"GT cars: {len(gt_boxes)}\n")

print("Model 1 (Normal):")
print(f"  TP={p1_tp}, FP={p1_fp}, FN={p1_fn}")
print(f"   Recall={p1_r:.3f}\n")

print("Model 2 (With SAHI):")
print(f"  TP={p2_tp}, FP={p2_fp}, FN={p2_fn}")
print(f"   Recall={p2_r:.3f}")

print("\n✔ Image 2 comparison complete")



=========== IMAGE 2 ANALYSIS ===========

GT cars: 22

Model 1 (Normal):
  TP=13, FP=33, FN=9
   Recall=0.591

Model 2 (With SAHI):
  TP=16, FP=142, FN=6
   Recall=0.727

✔ Image 2 comparison complete


## 👨‍💻 About Labellerr's Hands-On Learning in Computer Vision

Thank you for exploring this **Labellerr Hands-On Computer Vision Cookbook**! We hope this notebook helped you learn, prototype, and accelerate your vision projects.  
Labellerr provides ready-to-run Jupyter/Colab notebooks for the latest models and real-world use cases in computer vision, AI agents, and data annotation.

---
## 🧑‍🔬 Check Our Popular Youtube Videos

Whether you're a beginner or a practitioner, our hands-on training videos are perfect for learning custom model building, computer vision techniques, and applied AI:

- [How to Fine-Tune YOLO on Custom Dataset](https://www.youtube.com/watch?v=pBLWOe01QXU)  
  Step-by-step guide to fine-tuning YOLO for real-world use—environment setup, annotation, training, validation, and inference.
- [Build a Real-Time Intrusion Detection System with YOLO](https://www.youtube.com/watch?v=kwQeokYDVcE)  
  Create an AI-powered system to detect intruders in real time using YOLO and computer vision.
- [Finding Athlete Speed Using YOLO](https://www.youtube.com/watch?v=txW0CQe_pw0)  
  Estimate real-time speed of athletes for sports analytics.
- [Object Counting Using AI](https://www.youtube.com/watch?v=smsjBBQcIUQ)  
  Learn dataset curation, annotation, and training for robust object counting AI applications.
---

## 🎦 Popular Labellerr YouTube Videos

Level up your skills and see video walkthroughs of these tools and notebooks on the  
[Labellerr YouTube Channel](https://www.youtube.com/@Labellerr/videos):

- [How I Fixed My Biggest Annotation Nightmare with Labellerr](https://www.youtube.com/watch?v=hlcFdiuz_HI) – Solving complex annotation for ML engineers.
- [Explore Your Dataset with Labellerr's AI](https://www.youtube.com/watch?v=LdbRXYWVyN0) – Auto-tagging, object counting, image descriptions, and dataset exploration.
- [Boost AI Image Annotation 10X with Labellerr's CLIP Mode](https://www.youtube.com/watch?v=pY_o4EvYMz8) – Refine annotations with precision using CLIP mode.
- [Boost Data Annotation Accuracy and Efficiency with Active Learning](https://www.youtube.com/watch?v=lAYu-ewIhTE) – Speed up your annotation workflow using Active Learning.

> 👉 **Subscribe** for Labellerr's deep learning, annotation, and AI tutorials, or watch videos directly alongside notebooks!

---

## 🤝 Stay Connected

- **Website:** [https://www.labellerr.com/](https://www.labellerr.com/)
- **Blog:** [https://www.labellerr.com/blog/](https://www.labellerr.com/blog/)
- **GitHub:** [Labellerr/Hands-On-Learning-in-Computer-Vision](https://github.com/Labellerr/Hands-On-Learning-in-Computer-Vision)
- **LinkedIn:** [Labellerr](https://in.linkedin.com/company/labellerr)
- **Twitter/X:** [@Labellerr1](https://x.com/Labellerr1)

*Happy learning and building with Labellerr!*
